# Triviality of Inhomogeneous Deformations of $\mathfrak{osp}(1|2n)$

This notebook demonstrates the computational verification of the main theorem:

> **Theorem.** For $n \geq 2$, the $\gamma$-deformation of $B(0,n) = \mathfrak{osp}(1|2n)$ is trivial
> if and only if all deformation parameters vanish.

We verify this using:
1. **Certificate verification**: Algebraic proof that $c^\top A = 0$ and $c^\top L \neq 0$
2. **Triviality checking**: Numerical verification of $\delta f = \gamma$
3. **Dimension formula**: $\dim C^1_\mu = 6n - 2$ for weight sectors $\mu = \pm e_j$

For full details, see the [paper](https://arxiv.org/) and the [repository](https://github.com/ritsumei-aoi/osp-triviality).

## Setup


In [ ]:
import sys
import json
from fractions import Fraction
from pathlib import Path
from collections import defaultdict

# Ensure src/ and scripts/ are on the path
root = Path('.').resolve()
if str(root / 'src') not in sys.path:
    sys.path.insert(0, str(root / 'src'))
if str(root / 'scripts') not in sys.path:
    sys.path.insert(0, str(root / 'scripts'))

from verify_certificates_algebraic import (
    load_structure_constants, load_gamma_constants,
    get_family_I_certificate, get_family_II_certificate,
    get_family_III_certificate_minus, get_family_III_certificate_plus,
    verify_certificate, verify_all_certificates
)
from oscillator_lie_superalgebras.triviality_checker import check_triviality

print('All modules loaded successfully.')

## 1. Algebra Structure of $\mathfrak{osp}(1|2n)$

The oscillator Lie superalgebra $B(0,n) = \mathfrak{osp}(1|2n)$ has:
- Even part $\mathfrak{g}_{\bar{0}} \cong \mathfrak{sp}(2n)$ with dimension $2n^2 + n$
- Odd part $\mathfrak{g}_{\bar{1}}$ with dimension $2n$

The even part is spanned by Cartan elements $H_j$ and root vectors $E_{\alpha}^\pm$.
The odd part is spanned by $E_{\delta_j}^\pm$ ($j = 1, \ldots, n$).

Let's examine the structure for $n = 2$ and $n = 3$.

In [ ]:
for n_val in [2, 3]:
    with open(f'data/algebra_structures/B_0_{n_val}_structure.json') as f:
        alg_data = json.load(f)

    even = alg_data['basis']['even']
    odd = alg_data['basis']['odd']
    print(f'B(0,{n_val}) = osp(1|{2*n_val})')
    print(f'  Even basis ({len(even)} elements): {even}')
    print(f'  Odd basis ({len(odd)} elements): {odd}')
    print(f'  Total dimension: {len(even) + len(odd)}')
    print(f'  Expected (2n²+n | 2n) = ({2*n_val**2+n_val} | {2*n_val})')
    print()

## 2. Triviality Check

The coboundary equation $\delta f = \gamma$ decomposes by weight into:
$$A_\mu \cdot \mathbf{f}_\mu = L_\mu \cdot \mathbf{gb}_\mu$$

where $A_\mu$ is the coboundary matrix, $L_\mu$ is the deformation matrix,
and $\mathbf{gb}_\mu$ is the vector of deformation parameters in weight sector $\mu$.

### Exceptional case: $n = 1$ (always trivial)

For $B(0,1) = \mathfrak{osp}(1|2)$, the coboundary equation always has a solution,
so every deformation is trivial.

In [ ]:
print('=== B(0,1) = osp(1|2): Exceptional case ===')
for gb_values in [[1.0, -0.5], [0.3, 0.7], [2.0, -1.0]]:
    result = check_triviality(n=1, B=gb_values)
    status = '✓ TRIVIAL' if result['is_trivial'] else '✗ NON-TRIVIAL'
    print(f'  gb = {gb_values} → {status} (residual: {result["residual_norm"]:.2e})')

print('\nAll deformations are trivial regardless of parameter values.')

### General case: $n \geq 2$ (trivial iff $\mathbf{gb} = 0$)

For $n \geq 2$, the deformation is trivial only when all parameters vanish.

In [ ]:
print('=== B(0,2) = osp(1|4): General case ===')
test_cases = [
    ([0, 0, 0, 0], 'gb = 0'),
    ([1, 0, 0, 0], 'gb₁⁺ ≠ 0'),
    ([0, 1, 0, 0], 'gb₂⁺ ≠ 0'),
    ([0, 0, 1, 0], 'gb₁⁻ ≠ 0'),
    ([0, 0, 0, 1], 'gb₂⁻ ≠ 0'),
    ([0.5, -0.3, 1.0, 0.7], 'all nonzero'),
]

for gb_values, label in test_cases:
    result = check_triviality(n=2, B=gb_values)
    status = '✓ TRIVIAL' if result['is_trivial'] else '✗ NON-TRIVIAL'
    print(f'  {label:20s} → {status} (residual: {result["residual_norm"]:.2e})')

print('\n→ Only gb = 0 gives a trivial deformation.')

## 3. Certificate Verification

A **certificate** is a vector $c$ in the left null space of $A_\mu$ such that $c^\top L_\mu \neq 0$.
Its existence proves that $\delta f = \gamma$ has no solution when the corresponding
deformation parameter is nonzero.

Three certificate families cover all $2n$ deformation parameters:

| Family | Parameters covered | Index range |
|--------|-------------------|-------------|
| I | $\mathrm{gb}_{a_0, b_j^+}$ | $j = 1, \ldots, n-1$ |
| II | $\mathrm{gb}_{a_0, b_j^-}$ | $j = 1, \ldots, n-1$ |
| III | $\mathrm{gb}_{a_0, b_n^\pm}$ | (two certificates) |

### Full verification for $n = 2, 3, 4$

In [ ]:
for n_val in [2, 3, 4]:
    verify_all_certificates(0, n_val)

## 4. Detailed Certificate Inspection

Let's examine individual certificates for $n = 2$ to see the algebraic structure.

Each certificate is a linear combination of coboundary evaluations
$(\delta f)(X, Y)|_Z$ that lies in $\ker A_\mu^\top$ but not in $\ker L_\mu^\top$.

In [ ]:
n = 2
basis, parity, brackets = load_structure_constants(0, n)
gb_vars, gamma = load_gamma_constants(0, n)

print(f'B(0,{n}): {len(basis)} basis elements, {len(gb_vars)} deformation parameters')
print(f'Parameters: {gb_vars}')
print()

# Family I, j=1
cert_I = get_family_I_certificate(1, n)
print('=== Family I certificate (j=1) ===')
print('Components:')
for X, Y, Z, coeff in cert_I:
    print(f'  {coeff:>3} · (δf)({X}, {Y})|_{Z}')

ok, ctL = verify_certificate(cert_I, basis, parity, brackets, gamma, gb_vars, 'Family I, j=1')
print(f'\nVerification:')
print(f'  c^T A = 0: {ok}')
print(f'  c^T L = {dict(ctL)}')
print(f'  → Forces gb_a0_b1p = 0 ✓')

In [ ]:
# Family III certificates
for label, getter in [('III⁺', get_family_III_certificate_plus),
                       ('III⁻', get_family_III_certificate_minus)]:
    cert = getter(n)
    print(f'=== Family {label} certificate (n={n}) ===')
    print('Components:')
    for X, Y, Z, coeff in cert:
        print(f'  {coeff:>3} · (δf)({X}, {Y})|_{Z}')

    ok, ctL = verify_certificate(cert, basis, parity, brackets, gamma, gb_vars, f'Family {label}')
    print(f'\nVerification:')
    print(f'  c^T A = 0: {ok}')
    print(f'  c^T L = {dict(ctL)}')
    print()

## 5. Dimension Formula

The dimension of the $f$-variable space in each weight sector $\mu = \pm e_j$ is:
$$\dim C^1_\mu = 6n - 2$$

This is derived from an 8-category decomposition of the contributing
basis pairs $(\text{src}, \text{dst})$ with mixed parity and weight difference $\mu$.

In [ ]:
from verify_dim_formula import count_by_formula, get_fvar_sectors, load_structure, get_gb_weights

print('Dimension formula verification: dim C^1_μ = 6n - 2')
print(f'{"n":>3} {"Formula":>10} {"Computed":>10} {"Match":>6}')
print('-' * 35)

for n_val in range(1, 6):
    formula_count, _, _ = count_by_formula(n_val)
    expected = 6 * n_val - 2
    match = '✓' if formula_count == expected else '✗'
    print(f'{n_val:>3} {expected:>10} {formula_count:>10} {match:>6}')

In [ ]:
# Cross-check with actual structure data for n=2,3
print('Cross-check with actual algebra structure data:')
print()

for n_val in [2, 3]:
    structure = load_structure(0, n_val)
    if structure is None:
        continue
    sectors = get_fvar_sectors(structure, n_val)
    gb_weights = get_gb_weights(n_val)
    
    print(f'B(0,{n_val}): gb weight sectors')
    for mu in gb_weights:
        count = len(sectors.get(mu, []))
        expected = 6 * n_val - 2
        match = '✓' if count == expected else '✗'
        print(f'  μ = {mu}: dim = {count} (expected {expected}) {match}')
    print()

## Summary

| Property | $n = 1$ | $n \geq 2$ |
|----------|---------|------------|
| Triviality | Always trivial | Trivial iff $\mathrm{gb} = 0$ |
| Certificates | Not needed | $2n$ certificates (Families I, II, III) |
| $\dim C^1_\mu$ | $4$ | $6n - 2$ |
| Rank condition | $\operatorname{rank}([A|L]) = \operatorname{rank}(A)$ | $\operatorname{rank}([A|L]) > \operatorname{rank}(A)$ |

### Explore further

```python
# Check triviality for B(0,3) with specific parameters
result = check_triviality(n=3, B=[1, 0, 0, 0, 0, 0])

# Verify certificates for larger n (n=5 takes a few seconds)
verify_all_certificates(0, 5)
```

See the [scripts/](../scripts/) directory for the full implementation.